In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Multi Language Fashion Retrieval - Build & Fine-tune Model

## This notebook builds the complete Multi-Language fashion retrieval system and fine-tunes it on our JSONL dataset.

 ### Pipeline:
## 1. **Build Model Architecture** (AraBERT + BLIP-2 + Projections)
## 2. **Load & Process JSONL Data** (21k fashion items)
## 3. **Setup Training Pipeline** (Contrastive Loss + Optimization)
## 4. **Fine-tune Model** (2-4 hours training)
## 5. **Save Trained Model** (Ready for inference)


In [2]:
!pip install langdetect

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoModel, AutoTokenizer, Blip2Model, Blip2Processor, BlipImageProcessor, Blip2VisionModel
from transformers import get_linear_schedule_with_warmup
from langdetect import detect, LangDetectException

import json
import re
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time
from typing import List, Union, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" - GPU: {torch.cuda.get_device_name(0)}")
    print(f" - GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=1348cdfff9858693f92a6f021ffb6fe955b05accff8e04a45089330a28988824
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
PyTorch version: 2.6.0+cu124
CUDA available: True
 - GPU: Tesla T4
 - GPU Memory: 15.8 GB


### 1. Model Architecture


In [3]:
BLIP2_PATH = '/content/drive/MyDrive/RecoMind/Raw Models/blip2-opt-2.7b'
ARABERT_PATH = '/content/drive/MyDrive/RecoMind/Raw Models/arabertv2'
BERT_EN_PATH = '/content/drive/MyDrive/RecoMind/Raw Models/bert-base-uncased'

class MultilingualFashionRetrieval(nn.Module):
    def __init__(self, embed_dim=512, dropout=0.1):
        super().__init__()

        self.ar_text_tokenizer = AutoTokenizer.from_pretrained(ARABERT_PATH)
        self.ar_text_encoder = AutoModel.from_pretrained(ARABERT_PATH)

        self.en_text_tokenizer = AutoTokenizer.from_pretrained(BERT_EN_PATH)
        self.en_text_encoder = AutoModel.from_pretrained(BERT_EN_PATH)

        self.blip2_processor = Blip2Processor.from_pretrained(BLIP2_PATH)
        self.blip2_model = Blip2Model.from_pretrained(BLIP2_PATH).to(dtype=torch.float32)
        self.blip2_model.language_model = None

        for param in self.ar_text_encoder.parameters():
            param.requires_grad = False
        for param in self.en_text_encoder.parameters():
            param.requires_grad = False
        for param in self.blip2_model.parameters():
            param.requires_grad = False

        ar_text_dim = self.ar_text_encoder.config.hidden_size
        en_text_dim = self.en_text_encoder.config.hidden_size
        blip2_dim = self.blip2_model.config.qformer_config.hidden_size

        self.ar_text_projection = nn.Sequential(
            nn.Linear(ar_text_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout)
        )

        self.en_text_projection = nn.Sequential(
            nn.Linear(en_text_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout)
        )

        self.image_projection = nn.Sequential(
            nn.Linear(blip2_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout)
        )

        self.fusion_layer = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim)
        )

        self.embed_dim = embed_dim
        self._init_weights()

        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,} ({trainable_params / total_params:.1%})")
        print(f"Frozen parameters: {total_params - trainable_params:,}")

    def _init_weights(self):
        for module in [self.ar_text_projection, self.en_text_projection,
                       self.image_projection, self.fusion_layer]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

    def detect_language(self, text: str) -> str:
        try:
            arabic_pattern = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]')
            if arabic_pattern.search(text):
                return 'ar'
            detected = detect(text)
            return 'ar' if detected == 'ar' else 'en'
        except (LangDetectException, Exception):
            arabic_chars = len(re.findall(r'[\u0600-\u06FF]', text))
            total_chars = len(re.sub(r'\s+', '', text))
            return 'ar' if total_chars > 0 and arabic_chars / total_chars > 0.3 else 'en'

    def encode_text(self, texts: List[str]) -> torch.Tensor:
        if not texts:
            return torch.empty(0, self.embed_dim, device=self.device)

        ar_texts = []
        en_texts = []
        ar_indices = []
        en_indices = []

        for i, text in enumerate(texts):
            lang = self.detect_language(text)
            if lang == 'ar':
                ar_texts.append(text)
                ar_indices.append(i)
            else:
                en_texts.append(text)
                en_indices.append(i)

        all_embeds = torch.zeros(len(texts), self.embed_dim, device=self.device)

        if ar_texts:
            ar_inputs = self.ar_text_tokenizer(
                ar_texts, padding=True, truncation=True,
                max_length=128, return_tensors='pt'
            )
            ar_inputs = {k: v.to(self.device) for k, v in ar_inputs.items()}

            with torch.no_grad():
                ar_outputs = self.ar_text_encoder(**ar_inputs)
                ar_features = ar_outputs.last_hidden_state[:, 0, :]

            ar_embeds = self.ar_text_projection(ar_features)
            ar_embeds = F.normalize(ar_embeds, dim=-1)

            for i, embed in enumerate(ar_embeds):
                all_embeds[ar_indices[i]] = embed

        if en_texts:
            en_inputs = self.en_text_tokenizer(
                en_texts, padding=True, truncation=True,
                max_length=128, return_tensors='pt'
            )
            en_inputs = {k: v.to(self.device) for k, v in en_inputs.items()}

            with torch.no_grad():
                en_outputs = self.en_text_encoder(**en_inputs)
                en_features = en_outputs.last_hidden_state[:, 0, :]

            en_embeds = self.en_text_projection(en_features)
            en_embeds = F.normalize(en_embeds, dim=-1)

            for i, embed in enumerate(en_embeds):
                all_embeds[en_indices[i]] = embed

        return all_embeds

    def encode_text_batch(self, input_ids, attention_mask, lang):
        input_ids = input_ids.to(self.device)
        attention_mask = attention_mask.to(self.device)

        if lang == 'ar':
            outputs = self.ar_text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            features = outputs.last_hidden_state[:, 0, :]
            projected = self.ar_text_projection(features)
        else:
            outputs = self.en_text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            features = outputs.last_hidden_state[:, 0, :]
            projected = self.en_text_projection(features)

        return F.normalize(projected, dim=-1)

    def encode_image(self, images: Union[List[Image.Image], torch.Tensor]) -> torch.Tensor:
        if isinstance(images, list) and isinstance(images[0], Image.Image):
            inputs = self.blip2_processor(images=images, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(self.device)
        else:
            pixel_values = images.to(self.device)

        with torch.no_grad():
            vision_outputs = self.blip2_model.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs.last_hidden_state

            query_tokens = self.blip2_model.query_tokens.expand(image_embeds.shape[0], -1, -1)
            query_outputs = self.blip2_model.qformer(
                query_embeds=query_tokens,
                encoder_hidden_states=image_embeds,
                encoder_attention_mask=torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=self.device)
            )
            features = query_outputs.last_hidden_state.mean(dim=1)

        image_embeds = self.image_projection(features)
        return F.normalize(image_embeds, dim=-1)

    def encode_multimodal(self, texts: List[str], images: Union[List[Image.Image], torch.Tensor]) -> torch.Tensor:
        text_embeds = self.encode_text(texts)
        image_embeds = self.encode_image(images)

        combined = torch.cat([text_embeds, image_embeds], dim=-1)
        multimodal_embeds = self.fusion_layer(combined)
        return F.normalize(multimodal_embeds, dim=-1)

    @property
    def device(self):
        return next(self.parameters()).device

### Initialize model

In [4]:
print("Initializing model...")
torch.cuda.empty_cache()
model = MultilingualFashionRetrieval(embed_dim=512, dropout=0.1)
torch.cuda.empty_cache()
model = model.to(device)
print("Model ready!")

Initializing model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Total parameters: 1,340,518,400
Trainable parameters: 2,759,680 (0.2%)
Frozen parameters: 1,337,758,720
Model ready!


### 2. Dataset & Data Loading

In [5]:
class MultilingualFashionDataset(Dataset):
    def __init__(self, jsonl_path: str, image_dir: str,
                 tokenizer_ar, tokenizer_en,
                 image_processor,
                 max_length: int = 256,
                 max_samples: int = None):
        self.image_dir = image_dir
        self.tokenizer_ar = tokenizer_ar
        self.tokenizer_en = tokenizer_en
        self.image_processor = image_processor
        self.max_length = max_length

        print(f"Loading data from: {jsonl_path}")

        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                try:
                    item = json.loads(line.strip())
                    self.data.append(item)
                    if max_samples and len(self.data) >= max_samples:
                        break
                except json.JSONDecodeError:
                    print(f"  Skipping invalid JSON at line {line_num + 1}")
                    continue

        print(f"Loaded {len(self.data)} samples")
        self._validate_data()

    def _validate_data(self):
        print("Validating data...")
        missing_images = 0
        valid_samples = []

        for i, item in enumerate(self.data):
            if 'image' not in item or 'text' not in item or 'text_en' not in item:
                print(f"  Missing fields in item {i}: {item}")
                continue

            image_path = os.path.join(self.image_dir, item['image'])
            if not os.path.exists(image_path):
                missing_images += 1
                if missing_images <= 5:
                    print(f" Missing image: {image_path}")
                continue

            valid_samples.append(item)

        self.data = valid_samples
        print(f"Valid samples: {len(self.data)}")
        if missing_images > 0:
            print(f"Missing images: {missing_images}")

        if len(self.data) > 0:
            print(f"\n Sample data:")
            for i in range(min(3, len(self.data))):
                print(f"{i+1}. Image: {self.data[i]['image']}")
                print(f"Text (AR): {self.data[i]['text'][:100]}...")
                print(f"Text (EN): {self.data[i]['text_en'][:100]}...")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        image_path = os.path.join(self.image_dir, item['image'])
        try:
            image = Image.open(image_path).convert('RGB')
            pixel_values = self.image_processor(image, return_tensors='pt')['pixel_values'].squeeze(0)
        except Exception as e:
            print(f" Error loading image {image_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            pixel_values = self.image_processor(image, return_tensors='pt')['pixel_values'].squeeze(0)

        text_ar = item['text']
        text_en = item['text_en']

        text_inputs_ar = self.tokenizer_ar(
            text_ar, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )

        text_inputs_en = self.tokenizer_en(
            text_en, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )

        return {
            'pixel_values': pixel_values,
            'input_ids_ar': text_inputs_ar['input_ids'].squeeze(0),
            'attention_mask_ar': text_inputs_ar['attention_mask'].squeeze(0),
            'input_ids_en': text_inputs_en['input_ids'].squeeze(0),
            'attention_mask_en': text_inputs_en['attention_mask'].squeeze(0),
            'text_ar': text_ar,
            'text_en': text_en,
            'item_id': item.get('id', idx)
        }

### 3. Training Configuration

In [6]:
CONFIG = {
    'jsonl_path': '/content/drive/MyDrive/RecoMind/Processed Data/dataset_cleaned.jsonl',
    'image_dir': '/content/drive/MyDrive/RecoMind/Raw Data/',
    'train_split': 0.9,
    'max_samples': None,

    'preprocessed_images_dir': '/content/drive/MyDrive/RecoMind/Processed Data/preprocessed_tensors/images',
    'preprocessed_texts_ar_dir': '/content/drive/MyDrive/RecoMind/Processed Data/preprocessed_tensors/texts_ar',
    'preprocessed_texts_en_dir': '/content/drive/MyDrive/RecoMind/Processed Data/preprocessed_tensors/texts_en',

    'batch_size': 32,
    'epochs': 10,
    'learning_rate': 1e-3,
    'weight_decay': 0.01,
    'warmup_ratio': 0.03,
    'gradient_clip': 1.0,

    'temperature': 0.07,

    'log_interval': 25,
    'save_interval_epoch': 1,
    'output_dir': '/content/drive/MyDrive/RecoMind/Main Model/checkpoints',
    'model_name': 'multilingual_fashion_retrieval'
}

print(" Training Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

os.makedirs(CONFIG['output_dir'], exist_ok=True)

 Training Configuration:
   jsonl_path: /content/drive/MyDrive/RecoMind/Processed Data/dataset_cleaned.jsonl
   image_dir: /content/drive/MyDrive/RecoMind/Raw Data/
   train_split: 0.9
   max_samples: None
   preprocessed_images_dir: /content/drive/MyDrive/RecoMind/Processed Data/preprocessed_tensors/images
   preprocessed_texts_ar_dir: /content/drive/MyDrive/RecoMind/Processed Data/preprocessed_tensors/texts_ar
   preprocessed_texts_en_dir: /content/drive/MyDrive/RecoMind/Processed Data/preprocessed_tensors/texts_en
   batch_size: 32
   epochs: 10
   learning_rate: 0.001
   weight_decay: 0.01
   warmup_ratio: 0.03
   gradient_clip: 1.0
   temperature: 0.07
   log_interval: 25
   save_interval_epoch: 1
   output_dir: /content/drive/MyDrive/RecoMind/Main Model/checkpoints
   model_name: multilingual_fashion_retrieval


### 4. Contrastive Loss

In [7]:
class ContrastiveLoss(nn.Module):

    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, image_embeds: torch.Tensor, text_embeds: torch.Tensor) -> torch.Tensor:
        batch_size = image_embeds.size(0)

        sim_i2t = torch.matmul(image_embeds, text_embeds.T) / self.temperature
        sim_t2i = torch.matmul(text_embeds, image_embeds.T) / self.temperature

        labels = torch.arange(batch_size, device=image_embeds.device, dtype=torch.long)

        loss_i2t = F.cross_entropy(sim_i2t, labels)
        loss_t2i = F.cross_entropy(sim_t2i, labels)

        return (loss_i2t + loss_t2i) / 2

criterion = ContrastiveLoss(temperature=CONFIG['temperature'])
print(f"Contrastive loss initialized (temperature: {CONFIG['temperature']})")

Contrastive loss initialized (temperature: 0.07)


### 5. Load and Split Data

In [16]:
print("Loading and splitting data...")

full_dataset = MultilingualFashionDataset(
    jsonl_path=CONFIG['jsonl_path'],
    image_dir=CONFIG['image_dir'],
    tokenizer_ar=model.ar_text_tokenizer,
    tokenizer_en=model.en_text_tokenizer,
    image_processor=model.blip2_processor,
    max_samples=None
)

train_size = int(CONFIG['train_split'] * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Dataset splits:")
print(f" - Training: {len(train_dataset)} samples")
print(f" - Validation: {len(val_dataset)} samples")

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print(f"Data loaders created:")
print(f" - Train batches: {len(train_loader)}")
print(f" - Val batches: {len(val_loader)}")

Loading and splitting data...
Loading data from: /content/drive/MyDrive/RecoMind/Processed Data/dataset_cleaned.jsonl
Loaded 21157 samples
Validating data...
Valid samples: 21157

 Sample data:
1. Image: /content/drive/MyDrive/RecoMind/Raw Data/clothes_desc/ac1a7dc14fad31d1795f2d19e9717ed9.jpg
Text (AR): الكافتان الوردي طول العجل المنسوج في خليط مع عنق وأزرار مخفية أسفل الجبهة. نير مزدوج الطبقات يواصل أ...
Text (EN): Pink Calf-length kaftan woven in a Tencel™ lyocell blend with a V-neck and concealed buttons down th...
2. Image: /content/drive/MyDrive/RecoMind/Raw Data/clothes_desc/937bef632cb4f3f5f97fbbed9a3d61c7.jpg
Text (AR): قميص برتقال برتقال خفيف في كشمير ناعم مع طوق يقف، الأكمام طويلة راجلان مع الأصفاد المربطة، والحافة ا...
Text (EN): Light Orange Rib-knit jumper in soft cashmere with a stand-up collar, long raglan sleeves with ribbe...
3. Image: /content/drive/MyDrive/RecoMind/Raw Data/clothes_desc/346aa1a684d3fbd619e9501e39c6d790.jpg
Text (AR): سراويل (بوكسر) السوداء ذات الخصر

### 6. Setup Training


In [9]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

total_steps = len(train_loader) * CONFIG['epochs']
warmup_steps = int(CONFIG['warmup_ratio'] * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Training setup:")
print(f" - Optimizer: AdamW (lr={CONFIG['learning_rate']})")
print(f" - Total steps: {total_steps:,}")
print(f" - Warmup steps: {warmup_steps:,}")
print(f" - Trainable parameters: {len(trainable_params):,}")

train_losses = []
val_losses = []
best_val_loss = float('inf')

Training setup:
 - Optimizer: AdamW (lr=0.001)
 - Total steps: 5,950
 - Warmup steps: 178
 - Trainable parameters: 18


### 7. Training Loop

In [10]:
def compute_recall(image_embeds, text_embeds, device, ks=[1,5,10]):
    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds = F.normalize(text_embeds, dim=-1)

    sim_matrix = image_embeds @ text_embeds.T

    recalls = {}

    for k in ks:
        topk_text = torch.topk(sim_matrix, k, dim=1).indices
        correct_i2t = (topk_text == torch.arange(sim_matrix.size(0), device=device).unsqueeze(1)).any(dim=1).float()
        recall_i2t = correct_i2t.mean().item()

        topk_image = torch.topk(sim_matrix.T, k, dim=1).indices
        correct_t2i = (topk_image == torch.arange(sim_matrix.size(0), device=device).unsqueeze(1)).any(dim=1).float()
        recall_t2i = correct_t2i.mean().item()

        recalls[f'Recall@{k}_ImageToText'] = recall_i2t
        recalls[f'Recall@{k}_TextToImage'] = recall_t2i

    return recalls

In [11]:
def save_epoch_checkpoint(model, optimizer, scheduler, epoch, avg_loss, config):
    checkpoint_name = f"epoch{epoch+1}_final.pt"
    checkpoint_path = os.path.join(config['output_dir'], checkpoint_name)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': avg_loss,
        'config': config
    }

    torch.save(checkpoint, checkpoint_path)
    print(f"Epoch checkpoint saved: {checkpoint_name}")

In [12]:
def train_epoch(model, train_loader, optimizer, scheduler, criterion, epoch):
    model.train()
    total_loss = 0
    num_batches = len(train_loader)

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}')

    for batch_idx, batch in enumerate(progress_bar):
        pixel_values = batch['pixel_values'].to(device)
        input_ids_ar = batch['input_ids_ar'].to(device)
        attention_mask_ar = batch['attention_mask_ar'].to(device)
        input_ids_en = batch['input_ids_en'].to(device)
        attention_mask_en = batch['attention_mask_en'].to(device)

        optimizer.zero_grad()

        image_embeds = model.encode_image(pixel_values)
        text_embeds_ar = model.encode_text_batch(input_ids_ar, attention_mask_ar, lang='ar')
        text_embeds_en = model.encode_text_batch(input_ids_en, attention_mask_en, lang='en')

        loss_ar = criterion(image_embeds, text_embeds_ar)
        loss_en = criterion(image_embeds, text_embeds_en)
        loss = (loss_ar + loss_en) / 2

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Avg': f'{total_loss/(batch_idx+1):.4f}',
            'LR': f'{scheduler.get_last_lr()[0]:.2e}',
            'GradNorm': f'{grad_norm:.2e}'
        })

        if batch_idx % CONFIG['log_interval'] == 0:
            print(f'   Batch {batch_idx + 1}/{num_batches} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | GradNorm: {grad_norm:.2e}')

    return total_loss / num_batches


def validate_epoch(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    num_batches = len(val_loader)

    all_image_embeds = []
    all_text_embeds = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validating'):
            pixel_values = batch['pixel_values'].to(device)
            input_ids_ar = batch['input_ids_ar'].to(device)
            attention_mask_ar = batch['attention_mask_ar'].to(device)
            input_ids_en = batch['input_ids_en'].to(device)
            attention_mask_en = batch['attention_mask_en'].to(device)

            image_embeds = model.encode_image(pixel_values)
            text_embeds_ar = model.encode_text_batch(input_ids_ar, attention_mask_ar, lang='ar')
            text_embeds_en = model.encode_text_batch(input_ids_en, attention_mask_en, lang='en')

            loss_ar = criterion(image_embeds, text_embeds_ar)
            loss_en = criterion(image_embeds, text_embeds_en)
            loss = (loss_ar + loss_en) / 2
            total_loss += loss.item()

            batch_text_embeds = (text_embeds_ar + text_embeds_en) / 2

            all_image_embeds.append(image_embeds)
            all_text_embeds.append(batch_text_embeds)

    all_image_embeds = torch.cat(all_image_embeds, dim=0)
    all_text_embeds = torch.cat(all_text_embeds, dim=0)

    recall_metrics = compute_recall(all_image_embeds, all_text_embeds, device)

    print(f" Validation Recall: {recall_metrics}")

    return total_loss / num_batches, recall_metrics

***TO CONTINUE TRAINING FROM AN EPOCH***

In [13]:
torch.cuda.empty_cache()

CHECKPOINT_PATH = "/content/drive/MyDrive/RecoMind/Main Model/checkpoints/epoch8_final.pt"
print(f"Loading checkpoint from {CHECKPOINT_PATH}")

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

torch.cuda.empty_cache()

completed_epoch = checkpoint['epoch']
start_epoch = checkpoint['epoch'] + 1
avg_loss = checkpoint['loss']
config = checkpoint['config']

print(f"Checkpoint loaded successfully!")
print(f"Completed epoch: {completed_epoch + 1}")
print(f"Resume training from epoch: {start_epoch + 1}")
print(f"Previous avg loss: {avg_loss:.4f}")

Loading checkpoint from /content/drive/MyDrive/RecoMind/Main Model/checkpoints/epoch8_final.pt
Checkpoint loaded successfully!
Completed epoch: 8
Resume training from epoch: 9
Previous avg loss: 0.5893


***WE RUN THIS AFTER EPOCH 6 ONLY (DONT RUN)***

In [ ]:
# current_lr = optimizer.param_groups[0]['lr']
# print(f"Current LR from checkpoint: {current_lr:.2e}")

# reduction_factor = 0.5
# new_lr = current_lr * reduction_factor

# for param_group in optimizer.param_groups:
#    param_group['lr'] = new_lr

# if hasattr(scheduler, 'base_lrs'):
#    scheduler.base_lrs = [new_lr for _ in scheduler.base_lrs]

# print(f"LR adjusted: {current_lr:.2e} -> {new_lr:.2e} (reduced by {(1-reduction_factor)*100:.0f}%)")

In [ ]:
print(" Starting Training!")
print("=" * 60)

start_time = time.time()

for epoch in range(start_epoch, CONFIG['epochs']):
    print(f"\n Epoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 40)

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, epoch)
    train_losses.append(train_loss)

    val_loss, recall_metrics = validate_epoch(model, val_loader, criterion)
    val_losses.append(val_loss)

    print(f"\n Epoch {epoch + 1} Results:")
    print(f"   Train Loss: {train_loss:.4f}")
    print(f"   Val Loss: {val_loss:.4f}")
    for metric_name, metric_value in recall_metrics.items():
        print(f"   {metric_name}: {metric_value:.4f}")

    if epoch % CONFIG['save_interval_epoch'] == 0 or val_loss < best_val_loss:
        save_epoch_checkpoint(model, optimizer, scheduler, epoch, val_loss, CONFIG)

        # if val_loss < best_val_loss:
        #     best_val_loss = val_loss
        #     best_checkpoint = {
        #         'epoch': epoch + 1,
        #         'model_state_dict': model.state_dict(),
        #         'optimizer_state_dict': optimizer.state_dict(),
        #         'scheduler_state_dict': scheduler.state_dict(),
        #         'train_loss': train_loss,
        #         'val_loss': val_loss,
        #         'recall_metrics': recall_metrics,
        #         'config': CONFIG
        #     }
        #     best_model_path = os.path.join(CONFIG['output_dir'], f"{CONFIG['model_name']}_best.pt")
        #     torch.save(best_checkpoint, best_model_path)
        #     print(f" Best model updated! Val Loss: {val_loss:.4f}")

training_time = time.time() - start_time
print(f"\n Training Complete!")
print(f"  Total training time: {training_time/3600:.2f} hours")
print(f" Best validation loss: {best_val_loss:.4f}")

TEST

In [18]:
model.eval()
n_batches = 67
all_true_sim_ar = []
all_rand_sim_ar = []
all_diag_sim_ar = []
all_off_diag_sim_ar = []

all_true_sim_en = []
all_rand_sim_en = []
all_diag_sim_en = []
all_off_diag_sim_en = []

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= n_batches:
            break

        images = batch['pixel_values'].to(model.device)
        texts_ar = batch['text_ar']
        texts_en = batch['text_en']

        image_embeds = model.encode_image(images)
        text_embeds_ar = model.encode_text(texts_ar)
        text_embeds_en = model.encode_text(texts_en)

        image_embeds = torch.nn.functional.normalize(image_embeds, dim=1)
        text_embeds_ar = torch.nn.functional.normalize(text_embeds_ar, dim=1)
        text_embeds_en = torch.nn.functional.normalize(text_embeds_en, dim=1)

        true_sim_ar = (image_embeds * text_embeds_ar).sum(dim=-1).mean().item()
        all_true_sim_ar.append(true_sim_ar)

        rand_text_embeds_ar = text_embeds_ar[torch.randperm(text_embeds_ar.size(0))]
        rand_sim_ar = (image_embeds * rand_text_embeds_ar).sum(dim=-1).mean().item()
        all_rand_sim_ar.append(rand_sim_ar)

        sim_matrix_ar = torch.matmul(image_embeds, text_embeds_ar.T)
        diagonal_sim_ar = sim_matrix_ar.diag().mean().item()
        off_diagonal_sim_ar = sim_matrix_ar[~torch.eye(sim_matrix_ar.size(0), dtype=torch.bool, device=model.device)].mean().item()
        all_diag_sim_ar.append(diagonal_sim_ar)
        all_off_diag_sim_ar.append(off_diagonal_sim_ar)

        true_sim_en = (image_embeds * text_embeds_en).sum(dim=-1).mean().item()
        all_true_sim_en.append(true_sim_en)

        rand_text_embeds_en = text_embeds_en[torch.randperm(text_embeds_en.size(0))]
        rand_sim_en = (image_embeds * rand_text_embeds_en).sum(dim=-1).mean().item()
        all_rand_sim_en.append(rand_sim_en)

        sim_matrix_en = torch.matmul(image_embeds, text_embeds_en.T)
        diagonal_sim_en = sim_matrix_en.diag().mean().item()
        off_diagonal_sim_en = sim_matrix_en[~torch.eye(sim_matrix_en.size(0), dtype=torch.bool, device=model.device)].mean().item()
        all_diag_sim_en.append(diagonal_sim_en)
        all_off_diag_sim_en.append(off_diagonal_sim_en)

avg_true_sim_ar = sum(all_true_sim_ar) / len(all_true_sim_ar)
avg_rand_sim_ar = sum(all_rand_sim_ar) / len(all_rand_sim_ar)
avg_diag_sim_ar = sum(all_diag_sim_ar) / len(all_diag_sim_ar)
avg_off_diag_sim_ar = sum(all_off_diag_sim_ar) / len(all_off_diag_sim_ar)

avg_true_sim_en = sum(all_true_sim_en) / len(all_true_sim_en)
avg_rand_sim_en = sum(all_rand_sim_en) / len(all_rand_sim_en)
avg_diag_sim_en = sum(all_diag_sim_en) / len(all_diag_sim_en)
avg_off_diag_sim_en = sum(all_off_diag_sim_en) / len(all_off_diag_sim_en)

print(f"Model Diagnostic Results (all batches):")
print("=" * 50)
print(f"ARABIC TEXT-IMAGE RETRIEVAL:")
print(f" - True pairs similarity: {avg_true_sim_ar:.3f}")
print(f" - Random pairs similarity: {avg_rand_sim_ar:.3f}")
print(f" - Gap: {avg_true_sim_ar - avg_rand_sim_ar:.3f}")
print(f" - Matrix diagonal: {avg_diag_sim_ar:.3f}")
print(f" - Matrix off-diagonal: {avg_off_diag_sim_ar:.3f}")
print()
print(f"ENGLISH TEXT-IMAGE RETRIEVAL:")
print(f" - True pairs similarity: {avg_true_sim_en:.3f}")
print(f" - Random pairs similarity: {avg_rand_sim_en:.3f}")
print(f" - Gap: {avg_true_sim_en - avg_rand_sim_en:.3f}")
print(f" - Matrix diagonal: {avg_diag_sim_en:.3f}")
print(f" - Matrix off-diagonal: {avg_off_diag_sim_en:.3f}")
print()
print(f"CROSS-LANGUAGE COMPARISON:")
print(f" - Arabic vs English gap difference: {(avg_true_sim_ar - avg_rand_sim_ar) - (avg_true_sim_en - avg_rand_sim_en):.3f}")
print(f" - Arabic true pairs vs English true pairs: {avg_true_sim_ar - avg_true_sim_en:.3f}")

Model Diagnostic Results (all batches):
ARABIC TEXT-IMAGE RETRIEVAL:
 - True pairs similarity: 0.473
 - Random pairs similarity: 0.022
 - Gap: 0.451
 - Matrix diagonal: 0.473
 - Matrix off-diagonal: -0.001

ENGLISH TEXT-IMAGE RETRIEVAL:
 - True pairs similarity: 0.642
 - Random pairs similarity: 0.019
 - Gap: 0.623
 - Matrix diagonal: 0.642
 - Matrix off-diagonal: 0.001

CROSS-LANGUAGE COMPARISON:
 - Arabic vs English gap difference: -0.172
 - Arabic true pairs vs English true pairs: -0.169


### 8. Training Analysis & Visualization


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', color='blue')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', color='blue')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss (Log Scale)')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n Training Summary:")
print(f"   Initial train loss: {train_losses[0]:.4f}")
print(f"   Final train loss: {train_losses[-1]:.4f}")
print(f"   Initial val loss: {val_losses[0]:.4f}")
print(f"   Best val loss: {min(val_losses):.4f}")
print(f"   Loss reduction: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")

### 9. Quick Evaluation Test


In [ ]:
print(" Quick Evaluation Test")
print("=" * 30)

best_checkpoint = torch.load(os.path.join(CONFIG['output_dir'], f"{CONFIG['model_name']}_best.pth"))
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

test_samples = []
for i, sample in enumerate(val_dataset):
    if i >= 5:
        break
    test_samples.append(sample)

print(f" Testing with {len(test_samples)} samples...")

with torch.no_grad():
    for i, sample in enumerate(test_samples):
        text_ar = sample['text_ar']
        text_en = sample['text_en']
        image_path = os.path.join(CONFIG['image_dir'], sample['image'])

        if os.path.exists(image_path):
            image = Image.open(image_path).convert('RGB')
            image_embed = model.encode_image([image])

            text_embed_ar = model.encode_text([text_ar])
            text_embed_en = model.encode_text([text_en])

            sim_ar = torch.matmul(text_embed_ar, image_embed.T).item()
            sim_en = torch.matmul(text_embed_en, image_embed.T).item()

            print(f"\nSample {i+1}: {sample['image']}")
            print(f"  Arabic Text: {text_ar[:60]}...")
            print(f"  English Text: {text_en[:60]}...")
            print(f"  Similarity (Arabic): {sim_ar:.3f}")
            print(f"  Similarity (English): {sim_en:.3f}")
        else:
            print(f"Sample {i+1}: Could not load image: {image_path}")

### 10. Save Final Model


In [21]:
final_model_path = os.path.join(CONFIG['output_dir'], 'multilingual_fashion_model_final.pth')

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'embed_dim': model.embed_dim,
        'architecture': 'MultilingualFashionRetrieval',
        'text_encoder_ar': 'aubmindlab/bert-base-arabertv2',
        'text_encoder_en': 'bert-base-uncased',
        'image_encoder': 'Salesforce/blip2-opt-2.7b',
    },
    'training_config': CONFIG,
    'training_stats': {
        'epochs_trained': 8,
        'final_train_loss': avg_loss
    }
}, final_model_path)

print(f"Final model saved to: {final_model_path}")

print("TRAINING COMPLETE - MULTILINGUAL FASHION RETRIEVAL MODEL")

print(f"\n FINAL RESULTS:")
print(f"    Model Architecture: AraBERT + Bert Base Uncased + BLIP-2 + Projections")
print(f"    Training Data: {len(full_dataset)} fashion items")
print(f"    Epochs Trained: 8")
print(f"    Best Val Loss: {avg_loss:.4f}")

print(f"\n SAVED FILES:")
print(f"    Final Model: {final_model_path}")

print(f"\n MODEL CAPABILITIES:")
print(f"     Arabic text understanding (AraBERT)")
print(f"     English text understanding (Bert Base Uncased)")
print(f"     Fashion image analysis (BLIP-2)")
print(f"     Cross-modal retrieval (Text ↔ Image)")
print(f"     Multimodal fusion")
print(f"     Embedding dimension: {model.embed_dim}")

 Final model saved to: /content/drive/MyDrive/RecoMind/Main Model/checkpoints/multilingual_fashion_model_final.pth
TRAINING COMPLETE - MULTILINGUAL FASHION RETRIEVAL MODEL

 FINAL RESULTS:
    Model Architecture: AraBERT + Bert Base Uncased + BLIP-2 + Projections
    Training Data: 21157 fashion items
    Epochs Trained: 8
    Best Val Loss: 0.5893

 SAVED FILES:
    Final Model: /content/drive/MyDrive/RecoMind/Main Model/checkpoints/multilingual_fashion_model_final.pth

 MODEL CAPABILITIES:
     Arabic text understanding (AraBERT)
     English text understanding (Bert Base Uncased)
     Fashion image analysis (BLIP-2)
     Cross-modal retrieval (Text ↔ Image)
     Multimodal fusion
     Embedding dimension: 512
